# Homework 4: CatBoost, LightGBM, XGBoost, and Ensemble Workflow

This notebook is designed for Kaggle Playground Series S6E4. It focuses on meaningful model variety using three strong boosting models: CatBoost, LightGBM, and XGBoost. The workflow balances model performance with computation time by using reasonable feature engineering, smaller hyperparameter searches, and weighted probability averaging instead of very slow full stacking as the main ensemble.

## Step 1: Imports and setup

In [4]:
import sys
!{sys.executable} -m pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached narwhals-2.20.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
    --------------------------------------- 2.1/100.2 MB 23.5 MB/s eta 0:00:05
   ---- ----------------------------------- 11.5/100.2 MB 38.0 MB/s eta 0:00:03
   ----------- ---------------------------- 29.9/100.2 MB 59.2 MB/s eta 0:00:02
   ---------------- ----------------------- 40.1/100.2 MB 55.4 MB/s eta 0:00:02
   --------------------- ------------------ 54.8/100.2 MB 60.2 MB/s eta 0:00:01
   --------------------------- ------------ 68.9/100.2 MB 61.1 MB/s eta 0:00:01
   ---------------------------------- ----- 86.0/100.2 MB 63.1 MB/s eta 0:00:01
   --------------------------------------  100.1/100.2 MB 65.2 MB/s eta 0:00:01
   ---------------------------------------- 100.2/100.2 MB 60.4 MB/s  0:00:01
Using cached graphviz

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, classification_report
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

fast_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [6]:
train = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\train.csv")
test = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\test.csv")
sample = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())
display(test.head())

Train shape: (630000, 21)
Test shape: (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


## Basic cleaning and target encoding

In [7]:
target = "Irrigation_Need"

train_clean = train.copy()
test_clean = test.copy()

test_ids = test_clean["id"] if "id" in test_clean.columns else sample.iloc[:, 0]

le = LabelEncoder()
y = le.fit_transform(train_clean[target])

X = train_clean.drop(columns=[target])
X_test_final = test_clean.copy()

print("Target classes:", le.classes_)
print("Target distribution:")
display(train_clean[target].value_counts())
display(train_clean[target].value_counts(normalize=True))

Target classes: ['High' 'Low' 'Medium']
Target distribution:


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64

## Feature engineering function


In [ ]:
def engineer_features(df):
    df_new = df.copy()
    
    if "id" in df_new.columns:
        df_new = df_new.drop(columns=["id"])
    
    numeric_cols = df_new.select_dtypes(include=["int64", "float64"]).columns.tolist()
    
    df_new["missing_count"] = df_new.isna().sum(axis=1)
    
    if len(numeric_cols) > 0:
        df_new["numeric_mean"] = df_new[numeric_cols].mean(axis=1)
        df_new["numeric_std"] = df_new[numeric_cols].std(axis=1)
        df_new["numeric_min"] = df_new[numeric_cols].min(axis=1)
        df_new["numeric_max"] = df_new[numeric_cols].max(axis=1)
        df_new["numeric_range"] = df_new["numeric_max"] - df_new["numeric_min"]
    
    for col in numeric_cols:
        if (df_new[col] >= 0).all():
            df_new[f"log1p_{col}"] = np.log1p(df_new[col])
        df_new[f"{col}_squared"] = df_new[col] ** 2
    
    selected_numeric = numeric_cols[:5]
    
    for i in range(len(selected_numeric)):
        for j in range(i + 1, len(selected_numeric)):
            col1 = selected_numeric[i]
            col2 = selected_numeric[j]
            
            df_new[f"{col1}_x_{col2}"] = df_new[col1] * df_new[col2]
            df_new[f"{col1}_plus_{col2}"] = df_new[col1] + df_new[col2]
            df_new[f"{col1}_minus_{col2}"] = df_new[col1] - df_new[col2]
            df_new[f"{col1}_div_{col2}"] = df_new[col1] / (df_new[col2] + 1e-6)
    
    return df_new

## Apply feature engineering and encode categorical variables

In [9]:
X_fe = engineer_features(X)
X_test_fe = engineer_features(X_test_final)

print("Original train feature shape:", X.shape)
print("Engineered train feature shape:", X_fe.shape)

full = pd.concat([X_fe, X_test_fe], axis=0)
full_encoded = pd.get_dummies(full, drop_first=False)

# Fill missing values using the median of each column
full_encoded = full_encoded.fillna(full_encoded.median(numeric_only=True))

X_encoded = full_encoded.iloc[:len(X_fe), :]
X_test_encoded = full_encoded.iloc[len(X_fe):, :]

print("Final encoded train shape:", X_encoded.shape)
print("Final encoded test shape:", X_test_encoded.shape)

Original train feature shape: (630000, 20)
Engineered train feature shape: (630000, 87)
Final encoded train shape: (630000, 111)
Final encoded test shape: (270000, 111)


## Train-validation split

In [10]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (504000, 111)
X_valid: (126000, 111)


## Helper functions for evaluation and submissions

In [11]:
results = []

def evaluate_model(model, model_name, X_train, y_train, X_valid, y_valid):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    
    # Some models may return shape (n, 1), so flatten just in case
    y_pred = np.array(y_pred).ravel()
    
    bal_acc = balanced_accuracy_score(y_valid, y_pred)
    acc = accuracy_score(y_valid, y_pred)
    f1_macro = f1_score(y_valid, y_pred, average="macro")
    
    results.append({
        "Model": model_name,
        "Validation Balanced Accuracy": bal_acc,
        "Validation Accuracy": acc,
        "Validation Macro F1": f1_macro
    })
    
    print("=" * 70)
    print(model_name)
    print("Balanced Accuracy:", bal_acc)
    print("Accuracy:", acc)
    print("Macro F1:", f1_macro)
    print()
    print(classification_report(y_valid, y_pred, target_names=le.classes_))
    
    return model


def create_submission(pred_encoded, filename):
    pred_encoded = np.array(pred_encoded).astype(int).ravel()
    pred_labels = le.inverse_transform(pred_encoded)
    
    submission = sample.copy()
    submission[target] = pred_labels
    submission.to_csv(filename, index=False)
    
    print(f"Saved: {filename}")
    display(submission.head())
    
    return submission

## Model 1, tuned LightGBM

LightGBM is usually the best speed-to-performance model for tabular competitions, so this is the first model to tune.

In [13]:
lgbm = LGBMClassifier(
    random_state=RANDOM_STATE,
    objective="multiclass",
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1
)

lgbm_params = {
    "n_estimators": [300, 500],
    "learning_rate": [0.03, 0.05, 0.08],
    "num_leaves": [31, 63],
    "max_depth": [-1, 10, 20],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_params,
    n_iter=10,
    scoring="balanced_accuracy",
    cv=fast_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

lgbm_search.fit(X_train, y_train)

print("Best LightGBM params:")
print(lgbm_search.best_params_)

best_lgbm = evaluate_model(
    lgbm_search.best_estimator_,
    "Tuned LightGBM",
    X_train,
    y_train,
    X_valid,
    y_valid
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best LightGBM params:
{'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Tuned LightGBM
Balanced Accuracy: 0.9716514765299816
Accuracy: 0.9834285714285714
Macro F1: 0.9629024218091478

              precision    recall  f1-score   support

        High       0.89      0.95      0.92      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.99      0.97      0.98     47815

    accuracy                           0.98    126000
   macro avg       0.95      0.97      0.96    126000
weighted avg       0.98      0.98      0.98    126000



##  Model 2, tuned XGBoost

XGBoost is a strong benchmark boosting model. The search is intentionally smaller to avoid long computation times.

In [14]:
xgb = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric="mlogloss",
    tree_method="hist",
    n_jobs=-1
)

xgb_params = {
    "n_estimators": [200, 300, 400],
    "max_depth": [4, 5, 6],
    "learning_rate": [0.05, 0.08],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=xgb_params,
    n_iter=10,
    scoring="balanced_accuracy",
    cv=fast_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

print("Best XGBoost params:")
print(xgb_search.best_params_)

best_xgb = evaluate_model(
    xgb_search.best_estimator_,
    "Tuned XGBoost",
    X_train,
    y_train,
    X_valid,
    y_valid
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best XGBoost params:
{'subsample': 0.8, 'n_estimators': 400, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.08, 'colsample_bytree': 1.0}
Tuned XGBoost
Balanced Accuracy: 0.9644971572545257
Accuracy: 0.9853492063492063
Macro F1: 0.9715753966614886

              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



## Model 3, tuned CatBoost

CatBoost is the third boosting model. It is meaningfully different from LightGBM and XGBoost, which is helpful for ensembling. To save time, the tuning search is smaller.

In [ ]:
cat = CatBoostClassifier(
    random_seed=RANDOM_STATE,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    verbose=0,
    thread_count=-1
)

cat_params = {
    "iterations": [300, 500],
    
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.08],
    "l2_leaf_reg": [1, 3, 5, 7]
}

cat_search = RandomizedSearchCV(
    estimator=cat,
    param_distributions=cat_params,
    n_iter=8,
    scoring="balanced_accuracy",
    cv=fast_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

cat_search.fit(X_train, y_train)

print("Best CatBoost params:")
print(cat_search.best_params_)

best_cat = evaluate_model(
    cat_search.best_estimator_,
    "Tuned CatBoost",
    X_train,
    y_train,
    X_valid,
    y_valid
)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best CatBoost params:
{'learning_rate': 0.08, 'l2_leaf_reg': 1, 'iterations': 300, 'depth': 8}
Tuned CatBoost
Balanced Accuracy: 0.9636434126400676
Accuracy: 0.9854761904761905
Macro F1: 0.9712839194698081

              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



## Compare individual models

In [16]:
results_df = pd.DataFrame(results).sort_values(
    "Validation Balanced Accuracy",
    ascending=False
)

display(results_df)

,Model,Validation Balanced Accuracy,Validation Accuracy,Validation Macro F1
0,Tuned LightGBM,0.971651,0.983429,0.962902
1,Tuned XGBoost,0.964497,0.985349,0.971575
2,Tuned CatBoost,0.963643,0.985476,0.971284


## Feature importance from all three boosting models

In [17]:
def show_feature_importance(model, model_name, feature_names, top_n=20):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    else:
        print(f"{model_name} does not expose feature_importances_.")
        return None
    
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False)
    
    print(f"Top {top_n} features for {model_name}:")
    display(importance_df.head(top_n))
    
    return importance_df

lgbm_importance = show_feature_importance(best_lgbm, "LightGBM", X_encoded.columns)
xgb_importance = show_feature_importance(best_xgb, "XGBoost", X_encoded.columns)
cat_importance = show_feature_importance(best_cat, "CatBoost", X_encoded.columns)

Top 20 features for LightGBM:


,feature,importance
8,Wind_Speed_kmh,2006
4,Temperature_C,1371
104,Mulching_Used_No,1366
6,Rainfall_mm,1318
5,Humidity,1249
10,Previous_Irrigation_mm,1177
1,Soil_Moisture,1144
90,Crop_Growth_Stage_Harvest,965
91,Crop_Growth_Stage_Sowing,888
16,numeric_range,845


Top 20 features for XGBoost:


,feature,importance
92,Crop_Growth_Stage_Vegetative,0.193158
1,Soil_Moisture,0.191620
89,Crop_Growth_Stage_Flowering,0.131775
4,Temperature_C,0.096838
104,Mulching_Used_No,0.077850
91,Crop_Growth_Stage_Sowing,0.066400
90,Crop_Growth_Stage_Harvest,0.048117
8,Wind_Speed_kmh,0.032304
65,Soil_Moisture_minus_Temperature_C,0.032248
15,numeric_max,0.025209


Top 20 features for CatBoost:


,feature,importance
89,Crop_Growth_Stage_Flowering,14.464385
92,Crop_Growth_Stage_Vegetative,13.855588
20,Soil_Moisture_squared,7.982173
1,Soil_Moisture,6.115631
104,Mulching_Used_No,5.884879
105,Mulching_Used_Yes,4.993604
19,log1p_Soil_Moisture,4.782864
34,Wind_Speed_kmh_squared,4.262265
90,Crop_Growth_Stage_Harvest,3.487273
26,Temperature_C_squared,3.431841


## Permutation importance



In [18]:
# Pick the current best model from validation results
best_model_name = results_df.iloc[0]["Model"]
print("Best validation model:", best_model_name)

if "LightGBM" in best_model_name:
    best_single_model = best_lgbm
elif "XGBoost" in best_model_name:
    best_single_model = best_xgb
else:
    best_single_model = best_cat

perm_result = permutation_importance(
    best_single_model,
    X_valid,
    y_valid,
    scoring="balanced_accuracy",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": X_valid.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values("importance_mean", ascending=False)

display(perm_importance_df.head(20))

Best validation model: Tuned LightGBM


,feature,importance_mean,importance_std
1,Soil_Moisture,0.264936,0.003045
104,Mulching_Used_No,0.159390,0.000776
8,Wind_Speed_kmh,0.141968,0.001092
90,Crop_Growth_Stage_Harvest,0.131629,0.002088
91,Crop_Growth_Stage_Sowing,0.110618,0.001229
4,Temperature_C,0.109915,0.001296
13,numeric_std,0.033498,0.000542
89,Crop_Growth_Stage_Flowering,0.016275,0.000367
92,Crop_Growth_Stage_Vegetative,0.014323,0.000334
6,Rainfall_mm,0.006007,0.000166


## Weighted probability averaging ensemble


In [ ]:
lgbm_probs = best_lgbm.predict_proba(X_valid)
xgb_probs = best_xgb.predict_proba(X_valid)
cat_probs = best_cat.predict_proba(X_valid)

ensemble_weight_sets = {
    "Equal weights": [1/3, 1/3, 1/3],
    "LightGBM-heavy": [0.50, 0.30, 0.20],
    "XGBoost-heavy": [0.30, 0.50, 0.20],
    "CatBoost-heavy": [0.30, 0.20, 0.50],
    "LightGBM-XGBoost heavy": [0.45, 0.45, 0.10]
}

ensemble_results = []

for name, weights in ensemble_weight_sets.items():
    ensemble_probs = (
        weights[0] * lgbm_probs +
        weights[1] * xgb_probs +
        weights[2] * cat_probs
    )
    
    ensemble_pred = np.argmax(ensemble_probs, axis=1)
    
    ensemble_results.append({
        "Model": f"Weighted Probability Ensemble: {name}",
        "LightGBM Weight": weights[0],
        "XGBoost Weight": weights[1],
        "CatBoost Weight": weights[2],
        "Validation Balanced Accuracy": balanced_accuracy_score(y_valid, ensemble_pred),
        "Validation Accuracy": accuracy_score(y_valid, ensemble_pred),
        "Validation Macro F1": f1_score(y_valid, ensemble_pred, average="macro")
    })

ensemble_results_df = pd.DataFrame(ensemble_results).sort_values(
    "Validation Balanced Accuracy",
    ascending=False
)

display(ensemble_results_df)

best_ensemble_row = ensemble_results_df.iloc[0]
results.append({
    "Model": best_ensemble_row["Model"],
    "Validation Balanced Accuracy": best_ensemble_row["Validation Balanced Accuracy"],
    "Validation Accuracy": best_ensemble_row["Validation Accuracy"],
    "Validation Macro F1": best_ensemble_row["Validation Macro F1"]
})

results_df = pd.DataFrame(results).sort_values(
    "Validation Balanced Accuracy",
    ascending=False
)

display(results_df)

,Model,LightGBM Weight,XGBoost Weight,CatBoost Weight,Validation Balanced Accuracy,Validation Accuracy,Validation Macro F1
1,Weighted Probability Ensemble: LightGBM-heavy,0.500000,0.300000,0.200000,0.968111,0.985373,0.970889
4,Weighted Probability Ensemble: LightGBM-XGBoos...,0.450000,0.450000,0.100000,0.968053,0.985468,0.971517
0,Weighted Probability Ensemble: Equal weights,0.333333,0.333333,0.333333,0.967036,0.985643,0.972109
2,Weighted Probability Ensemble: XGBoost-heavy,0.300000,0.500000,0.200000,0.966725,0.985516,0.971753
3,Weighted Probability Ensemble: CatBoost-heavy,0.300000,0.200000,0.500000,0.966364,0.985619,0.971909


,Model,Validation Balanced Accuracy,Validation Accuracy,Validation Macro F1
0,Tuned LightGBM,0.971651,0.983429,0.962902
3,Weighted Probability Ensemble: LightGBM-heavy,0.968111,0.985373,0.970889
1,Tuned XGBoost,0.964497,0.985349,0.971575
2,Tuned CatBoost,0.963643,0.985476,0.971284


## Retrain final models on all training data

In [20]:
final_lgbm = lgbm_search.best_estimator_
final_xgb = xgb_search.best_estimator_
final_cat = cat_search.best_estimator_

final_lgbm.fit(X_encoded, y)
final_xgb.fit(X_encoded, y)
final_cat.fit(X_encoded, y)

CatBoostClassifier(depth=8, eval_metric='TotalF1', iterations=300, l2_leaf_reg=1, learning_rate=0.08, loss_function='MultiClass', random_seed=42, verbose=0)

## Create individual Kaggle submissions

In [21]:
lgbm_test_pred = final_lgbm.predict(X_test_encoded)
xgb_test_pred = final_xgb.predict(X_test_encoded)
cat_test_pred = final_cat.predict(X_test_encoded)

create_submission(lgbm_test_pred, "hw4_lightgbm_submission.csv")
create_submission(xgb_test_pred, "hw4_xgboost_submission.csv")
create_submission(cat_test_pred, "hw4_catboost_submission.csv")

Saved: hw4_lightgbm_submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Saved: hw4_xgboost_submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Saved: hw4_catboost_submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Medium
269996,899996,Low
269997,899997,Medium
269998,899998,Low


##  Create final weighted ensemble submission

In [22]:
print("Best ensemble from validation:")
display(best_ensemble_row.to_frame().T)

best_weights = [
    best_ensemble_row["LightGBM Weight"],
    best_ensemble_row["XGBoost Weight"],
    best_ensemble_row["CatBoost Weight"]
]

lgbm_test_probs = final_lgbm.predict_proba(X_test_encoded)
xgb_test_probs = final_xgb.predict_proba(X_test_encoded)
cat_test_probs = final_cat.predict_proba(X_test_encoded)

final_ensemble_probs = (
    best_weights[0] * lgbm_test_probs +
    best_weights[1] * xgb_test_probs +
    best_weights[2] * cat_test_probs
)

final_ensemble_pred = np.argmax(final_ensemble_probs, axis=1)

create_submission(
    final_ensemble_pred,
    "hw4_catboost_lightgbm_xgboost_ensemble_submission.csv"
)

Best ensemble from validation:


,Model,LightGBM Weight,XGBoost Weight,CatBoost Weight,Validation Balanced Accuracy,Validation Accuracy,Validation Macro F1
1,Weighted Probability Ensemble: LightGBM-heavy,0.5,0.3,0.2,0.968111,0.985373,0.970889


Saved: hw4_catboost_lightgbm_xgboost_ensemble_submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Medium
269996,899996,Low
269997,899997,Medium
269998,899998,Low


##  Manually enter Kaggle leaderboard scores



In [ ]:
leaderboard_scores = pd.DataFrame({
    "Approach": [
        "LightGBM",
        "XGBoost",
        "CatBoost",
        "Weighted Probability Ensemble"
    ],
    "Validation Balanced Accuracy": [
        results_df.loc[results_df["Model"].str.contains("LightGBM", case=False), "Validation Balanced Accuracy"].max(),
        results_df.loc[results_df["Model"].str.contains("XGBoost", case=False), "Validation Balanced Accuracy"].max(),
        results_df.loc[results_df["Model"].str.contains("CatBoost", case=False), "Validation Balanced Accuracy"].max(),
        results_df.loc[results_df["Model"].str.contains("Weighted Probability Ensemble", case=False), "Validation Balanced Accuracy"].max()
    ],
    "Kaggle Leaderboard Score": [
        None,
        None,
        None,
        None
    ],
    "Notes": [
        "Fast and strong boosting model for tabular data",
        "Strong benchmark boosting model",
        "Different boosting algorithm, useful for ensemble diversity",
        "Combined predicted probabilities from all three boosting models"
    ]
})

display(leaderboard_scores)

For feature engineering, I tried creating extra features to help the model capture more patterns in the data. I added numeric summary features like row mean, standard deviation, minimum, and maximum, as well as missing-value counts, log transformations, squared features, interaction features, and ratio features. I tried using other means of adding more features like polynomialfeatures, but it was not effective only adding bloat, so I scrapped it. 

These features worked best with the tuned LightGBM model, which achieved the strongest results: 0.9717 validation balanced accuracy, 0.96739 public leaderboard, and 0.96951 private leaderboard.

Overall, feature engineering helped, but the improvements were not huge. It gave the model a small but meaningful boost, especially compared to the Random Forest baseline, which had 0.9669 validation balanced accuracy, 0.96211 public leaderboard, and 0.96419 private leaderboard. For the ensemble, it did not work as well as I had hoped, as the 3 models used probably were making similar tree based decisions, in the future I should explore other models to ensemble rather than those specfici 3. 